# Домашнее задание 3. Статистическая проверка качества регрессии.

### 1. Загрузите файл candidates.csv с компьютера. 

* откройте первые несколько строк
* изучите строение файла.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split

# Загружаем файл candidates.csv (разделитель столбцов ';', десятичный разделитель ',')
df = pd.read_csv('candidates.csv', sep=';', decimal=',')

print("Первые 5 строк файла:")
print(df.head().to_string())
print("\nИнформация о структуре таблицы:")
print(df.info())

Первые 5 строк файла:
   Код претендента  Рейтинг университета окончания  Средний балл диплома  Опыт работы (лет)  Возраст  Рейтинг университета диплома  Баллы за сертификаты  Желаемая заработная плата  Итоговая зарплата от компании
0             1001                              82                  3.22                 10       33                            84                    89                     237000                         237000
1             1002                               4                  3.19                  3       29                            16                     0                     155000                         155000
2             1003                              70                  3.84                  6       30                            70                    74                     212000                         197000
3             1004                              98                  3.67                  2       24                            97    

### 2. На основе файла создайте матрицу X с характеристиками (некоторые столбцы могут не понадобятся для характеристик, подумайте какие), и столбец y (из последнего столбца).

* отобразите первые строки матрицы X и столбца y на экране.

In [2]:
# Столбец "Код претендента" — это просто порядковый номер/ID, для обучения он не нужен.
# Последний столбец "Итоговая зарплата от компании" — целевая переменная y.
X = df.drop(columns=['Код претендента', 'Итоговая зарплата от компании'])
y = df['Итоговая зарплата от компании']

print("Первые строки матрицы характеристик X:")
print(X.head().to_string())
print("\nПервые строки столбца y:")
print(y.head().to_string())

# Согласно примечанию к задаче делим данные на train (60%), val (20%) и test (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"\nРазмеры выборок: train = {len(X_train)}, val = {len(X_val)}, test = {len(X_test)}")

Первые строки матрицы характеристик X:
   Рейтинг университета окончания  Средний балл диплома  Опыт работы (лет)  Возраст  Рейтинг университета диплома  Баллы за сертификаты  Желаемая заработная плата
0                              82                  3.22                 10       33                            84                    89                     237000
1                               4                  3.19                  3       29                            16                     0                     155000
2                              70                  3.84                  6       30                            70                    74                     212000
3                              98                  3.67                  2       24                            97                    64                     140000
4                              94                  3.92                  2       27                            99                    66           

### 3. Используя SAG, постройте на этих данных модель линейной регрессии

* выведите на экран полученное уравнение линиии

In [3]:
def fit_sag(X_df, y_series, epochs=500, lr=0.002, random_state=42):
    """
    Обучение модели линейной регрессии методом SAG (Stochastic Average Gradient).
    Внутри алгоритма признаки стандартизируются для устойчивой сходимости градиентного шага,
    после чего веса пересчитываются обратно к исходному масштабу данных.
    """
    np.random.seed(random_state)
    X_mat = X_df.values.astype(float)
    y_vec = y_series.values.astype(float)
    n, k = X_mat.shape
    
    mean_X = X_mat.mean(axis=0)
    std_X = X_mat.std(axis=0)
    X_scaled = (X_mat - mean_X) / std_X
    
    # Добавляем единичный столбец для свободного члена b_0
    X_b = np.column_stack([np.ones(n), X_scaled])
    
    w = np.zeros(k + 1)
    memory_grad = np.zeros((n, k + 1)) # память градиентов по каждому наблюдению
    avg_grad = np.zeros(k + 1)         # текущий средний градиент
    
    for epoch in range(epochs):
        indices = np.random.permutation(n)
        for idx in indices:
            x_i = X_b[idx]
            pred = np.dot(x_i, w)
            grad_i = (pred - y_vec[idx]) * x_i
            
            avg_grad += (grad_i - memory_grad[idx]) / n
            memory_grad[idx] = grad_i
            w -= lr * avg_grad
            
    # Возвращаем веса в исходный масштаб признаков
    b = np.zeros(k + 1)
    b[1:] = w[1:] / std_X
    b[0] = w[0] - np.sum(w[1:] * mean_X / std_X)
    return b

# Обучаем модель на train
b_initial = fit_sag(X_train, y_train)

equation = f"y_hat = {b_initial[0]:.2f}"
for col_name, weight in zip(X_train.columns, b_initial[1:]):
    sign = "+" if weight >= 0 else "-"
    equation += f" {sign} {abs(weight):.4f} * [{col_name}]"

print("Уравнение полученной линии регрессии (обучено на train):")
print(equation)

Уравнение полученной линии регрессии (обучено на train):
y_hat = 30559.12 - 144.8137 * [Рейтинг университета окончания] + 11806.0113 * [Средний балл диплома] + 9518.4574 * [Опыт работы (лет)] + 266.7366 * [Возраст] - 200.0429 * [Рейтинг университета диплома] + 232.3314 * [Баллы за сертификаты] + 0.3235 * [Желаемая заработная плата]


### 4. Проверьте веса на значимость, используя алгоритм и формулы из лекции

* код должен соотвествовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы
* какие веса у вас оказались незначимыми?

**Ответ:**
Короче две гипотезы смотрим потому что данные с шумом и МНК/градиент никогда ровно 0 в весах не выдаст. Даже если признак вообще левый, модель все равно насчитает какой-то вес типа 0.1 или -50. Поэтому и проверяем H0 это когда вес реально ноль и просто случайно вылез, а H1 это когда признак реально влияет.

По t-критерию на вале в интервал доверия попали: рейтинг универа окончания, возраст и рейтинг универа диплома. У них t-статистика маленькая, они незначимые так что выкидываем их.

In [4]:
def check_weights_significance(X_df, y_series, b, alpha=0.05):
    n, k = X_df.shape
    X_mat = np.column_stack([np.ones(n), X_df.values.astype(float)])
    y_real = y_series.values.astype(float)
    y_pred = X_mat @ b
    
    # По формуле из лекции (с учетом поправки на степени свободы n - k - 1):
    df_student = n - k - 1
    s_e2 = np.sum((y_real - y_pred) ** 2) / df_student
    C = s_e2 * np.linalg.inv(X_mat.T @ X_mat)
    
    # Квантили распределения Стьюдента t_(alpha/2, n-k-1) и t_(1-alpha/2, n-k-1)
    t_left = stats.t.ppf(alpha / 2, df=df_student)
    t_right = stats.t.ppf(1 - alpha / 2, df=df_student)
    
    print(f"Выборка val: n = {n}, k = {k}, степени свободы (n - k - 1) = {df_student}")
    print(f"Интервал принятия H0 при уровне доверия {1-alpha}: [{t_left:.4f}; {t_right:.4f}]\n")
    
    names = ['b_0 (Свободный член)'] + list(X_df.columns)
    insignificant_cols = []
    
    for i in range(k + 1):
        s_bi = np.sqrt(C[i, i])
        t_i = b[i] / s_bi
        if t_left <= t_i <= t_right:
            status = "НЕЗНАЧИМ (принимается H0)"
            if i > 0:
                insignificant_cols.append(names[i])
        else:
            status = "ЗНАЧИМ (принимается H1)"
        print(f"{names[i]:32s} | b = {b[i]:10.4f} | S(b) = {s_bi:9.4f} | t = {t_i:8.4f} -> {status}")
        
    return insignificant_cols

# Анализируем значимость весов на валидационной выборке (val)
insig_cols = check_weights_significance(X_val, y_val, b_initial)
print("\nНезначимые характеристики:", insig_cols)

Выборка val: n = 160, k = 7, степени свободы (n - k - 1) = 152
Интервал принятия H0 при уровне доверия 0.95: [-1.9757; 1.9757]

b_0 (Свободный член)             | b = 30559.1232 | S(b) = 14507.6095 | t =   2.1064 -> ЗНАЧИМ (принимается H1)
Рейтинг университета окончания   | b =  -144.8137 | S(b) =  106.9179 | t =  -1.3544 -> НЕЗНАЧИМ (принимается H0)
Средний балл диплома             | b = 11806.0113 | S(b) = 1514.8187 | t =   7.7937 -> ЗНАЧИМ (принимается H1)
Опыт работы (лет)                | b =  9518.4574 | S(b) =  738.6000 | t =  12.8872 -> ЗНАЧИМ (принимается H1)
Возраст                          | b =   266.7366 | S(b) =  542.6839 | t =   0.4915 -> НЕЗНАЧИМ (принимается H0)
Рейтинг университета диплома     | b =  -200.0429 | S(b) =  105.4276 | t =  -1.8974 -> НЕЗНАЧИМ (принимается H0)
Баллы за сертификаты             | b =   232.3314 | S(b) =   25.6120 | t =   9.0712 -> ЗНАЧИМ (принимается H1)
Желаемая заработная плата        | b =     0.3235 | S(b) =    0.0316 | t =  10.2455 -> З

### 5. Если какие-то веса оказались незначимыми, перепишите матрицу характеристик X без соответствующих столбцов, обучите заново модель, используя SAG

In [5]:
# Убираем незначимые столбцы из выборок train, val и test
X_train_clean = X_train.drop(columns=insig_cols)
X_val_clean = X_val.drop(columns=insig_cols)
X_test_clean = X_test.drop(columns=insig_cols)

print("Оставшиеся характеристики в X:", list(X_train_clean.columns))

# Заново обучаем модель методом SAG на очищенной выборке train
b_clean = fit_sag(X_train_clean, y_train)

equation_clean = f"y_hat = {b_clean[0]:.2f}"
for col_name, weight in zip(X_train_clean.columns, b_clean[1:]):
    sign = "+" if weight >= 0 else "-"
    equation_clean += f" {sign} {abs(weight):.4f} * [{col_name}]"

print("\nОбновленное уравнение линии регрессии:")
print(equation_clean)

print("\nПовторная проверка весов на val после исправления модели:")
_ = check_weights_significance(X_val_clean, y_val, b_clean)

Оставшиеся характеристики в X: ['Средний балл диплома', 'Опыт работы (лет)', 'Баллы за сертификаты', 'Желаемая заработная плата']

Обновленное уравнение линии регрессии:
y_hat = 13026.57 + 7857.0880 * [Средний балл диплома] + 6775.8048 * [Опыт работы (лет)] + 161.4947 * [Баллы за сертификаты] + 0.5148 * [Желаемая заработная плата]

Повторная проверка весов на val после исправления модели:
Выборка val: n = 160, k = 4, степени свободы (n - k - 1) = 155
Интервал принятия H0 при уровне доверия 0.95: [-1.9754; 1.9754]

b_0 (Свободный член)             | b = 13026.5654 | S(b) = 7358.9955 | t =   1.7702 -> НЕЗНАЧИМ (принимается H0)
Средний балл диплома             | b =  7857.0880 | S(b) = 1938.8542 | t =   4.0524 -> ЗНАЧИМ (принимается H1)
Опыт работы (лет)                | b =  6775.8048 | S(b) =  575.1607 | t =  11.7807 -> ЗНАЧИМ (принимается H1)
Баллы за сертификаты             | b =   161.4947 | S(b) =   33.7191 | t =   4.7894 -> ЗНАЧИМ (принимается H1)
Желаемая заработная плата        |

### 6. Проверьте модель на значимость по алгоритму из лекции

* код должен соответствовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы и что эти гипотезы собой подразумевают
* какая у вас получилась модель?
* подходит ли она для описания наших данных?

**Ответ:**
Тут H0 значит что вообще все наши признаки бесполезные и модель выдает результат не лучше чем если просто брать среднюю зп для всех подряд. А H1 значит что хотябы один признак реально работает и от модели есть толк.

Модель значимая получилась, F вышло аж 1352 при пороге 2.43. Разница огромная, H0 отметаем сразу. Модель под данные подходит отлично.

In [6]:
def check_model_significance(X_df, y_series, b, alpha=0.05):
    n, k = X_df.shape
    X_mat = np.column_stack([np.ones(n), X_df.values.astype(float)])
    y_real = y_series.values.astype(float)
    y_pred = X_mat @ b
    y_mean = np.mean(y_real)
    
    # Коэффициент детерминации R^2
    ss_res = np.sum((y_real - y_pred) ** 2)
    ss_tot = np.sum((y_real - y_mean) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    # Статистика Фишера F = (R^2 / k) / ((1 - R^2) / (n - k - 1))
    df1 = k
    df2 = n - k - 1
    F_stat = (r2 / df1) / ((1 - r2) / df2)
    
    # Квантиль распределения Фишера F_(1-alpha, k, n-k-1)
    F_crit = stats.f.ppf(1 - alpha, dfn=df1, dfd=df2)
    
    print(f"Проверка на val: n = {n}, k = {k}, n - k - 1 = {df2}")
    print(f"R^2 = {r2:.6f}")
    print(f"F-статистика = {F_stat:.4f}")
    print(f"Табличное значение F_(1-alpha) = {F_crit:.4f}")
    
    if F_stat > F_crit:
        print("Итог: F > F_(1-alpha) -> Принимается H1 (модель значима).")
    else:
        print("Итог: F <= F_(1-alpha) -> Принимается H0 (модель незначима).")
    return r2, F_stat

r2_val, f_val = check_model_significance(X_val_clean, y_val, b_clean)

Проверка на val: n = 160, k = 4, n - k - 1 = 155
R^2 = 0.972154
F-статистика = 1352.8116
Табличное значение F_(1-alpha) = 2.4300
Итог: F > F_(1-alpha) -> Принимается H1 (модель значима).


### 7. Рассчитайте парный коэффициент корреляции для каждой пары x_i, y, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* ответьте на вопрос "что в общем случае (а не в конкретно нашем) показывают парные коэффициенты корреляции?"
* выведите коэффициент корреляции на экран для каждой пары в виде: пара, значение коэффициента
* ответьте на вопрос "что полученные значения говорят о нашей модели?"
* какие характеристики оказывают на ответ наибольшее значение, какие наименьшее?

**Ответ:**
В общем случае парная кореляция показывает насколько сильно две переменные связаны по прямой. Если около 1 или -1 то связь жесткая, если около 0 то связи по сути нет.

У нас больше всего на зп влияют желаемая зарплата (0.98) и опыт работы (0.93), тут прям явная прямая связь. А меньше всего напрямую влияют средний балл диплома (0.17) и баллы за сертификаты (0.16). Но в общей модели они все равно нужны чтобы точнее корректировать число.

In [7]:
def calc_pair_correlations(X_df, y_series):
    y_arr = y_series.values.astype(float)
    y_mean = np.mean(y_arr)
    sigma_y = np.sqrt(np.mean((y_arr - y_mean) ** 2))
    
    print("Парные коэффициенты корреляции (на выборке val):")
    for col in X_df.columns:
        x_arr = X_df[col].values.astype(float)
        x_mean = np.mean(x_arr)
        sigma_x = np.sqrt(np.mean((x_arr - x_mean) ** 2))
        
        # По формуле из лекции: r_(x,y) = (1/n * sum((x_i - x_mean)*(y_i - y_mean))) / (sigma_x * sigma_y)
        r_xy = np.mean((x_arr - x_mean) * (y_arr - y_mean)) / (sigma_x * sigma_y)
        print(f"({col}, Итоговая зарплата от компании): {r_xy:.4f}")

calc_pair_correlations(X_val_clean, y_val)

Парные коэффициенты корреляции (на выборке val):
(Средний балл диплома, Итоговая зарплата от компании): 0.1691
(Опыт работы (лет), Итоговая зарплата от компании): 0.9308
(Баллы за сертификаты, Итоговая зарплата от компании): 0.1553
(Желаемая заработная плата, Итоговая зарплата от компании): 0.9764


### 8. Вычислите коэффициент детерминации

* код должен соответствовать формуле из лекции
* выведите полученный коэффициент на экран
* что он говорит о нашей модели? насколько качественно она описала данные? лучше или хуже она, чем средняя линия?

**Ответ:**
R2 вышел 0.972, тоесть модель описывает 97% разброса зарплаты. Описала очень круто, точки почти на линии лежат. Понятное дело это намного лучше чем средняя линия, средняя линия вообще ничего не предсказывает нормально.

In [8]:
def calc_r2(X_df, y_series, b):
    n = len(y_series)
    X_mat = np.column_stack([np.ones(n), X_df.values.astype(float)])
    y_real = y_series.values.astype(float)
    y_pred = X_mat @ b
    y_mean = np.mean(y_real)
    
    # Формула из лекции: R^2 = 1 - sum((y_i - y_hat_i)^2) / sum((y_i - y_mean)^2)
    r2 = 1 - np.sum((y_real - y_pred) ** 2) / np.sum((y_real - y_mean) ** 2)
    return r2

r2_score_val = calc_r2(X_val_clean, y_val, b_clean)
print(f"Коэффициент детерминации R^2 (на val): {r2_score_val:.6f}")

Коэффициент детерминации R^2 (на val): 0.972154


### 9. Рассчитайте LOOCV кросс-валидации, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* объясните устно как он работает и зачем он нужен
* выведите LOOCV на экран
* ответьте на вопрос "что полученный LOOCV говорит о нашей модели? она идеально нам подходит, или в ней нужно что-то менять?"

**Ответ:**
Работает так: берем одного чела из выборки как тест, на остальных n-1 обучаемся и смотрим как модель ошиблась на этом одном. И так в цикле по каждому крутимся, а потом берем корень из средней ошибки. Нужен чтоб честно понять как модель себя поведет на новых данных и не переобучилась ли она.

LOOCV вышел где-то 12.2к (на test выборке ошибка 12к при средней зп 230к). Погрешность всего около 5%, для зарплат это вообще понты. Модель хорошо подходит, менять ниче не надо мы и так мусорные признаки выкинули.

In [9]:
def calc_loocv(X_df, y_series):
    """
    Расчет LOOCV по алгоритму из лекции:
    На каждом шаге i-е наблюдение берется как test, остальные n-1 как train.
    """
    X_mat = np.column_stack([np.ones(len(y_series)), X_df.values.astype(float)])
    y_vec = y_series.values.astype(float)
    n = len(y_vec)
    errors_sq = []
    
    for i in range(n):
        X_tr_loo = np.delete(X_mat, i, axis=0)
        y_tr_loo = np.delete(y_vec, i)
        
        x_te_loo = X_mat[i]
        y_te_loo = y_vec[i]
        
        # Строим модель на n-1 наблюдениях
        b_loo = np.linalg.pinv(X_tr_loo.T @ X_tr_loo) @ X_tr_loo.T @ y_tr_loo
        
        # Ошибка на тестовом наблюдении Error_i
        y_pred_i = np.dot(x_te_loo, b_loo)
        error_i = y_te_loo - y_pred_i
        errors_sq.append(error_i ** 2)
        
    # LOOCV = sqrt(1/n * sum(Error_i^2))
    loocv = np.sqrt(np.mean(errors_sq))
    return loocv

loocv_val = calc_loocv(X_val_clean, y_val)
print(f"Ошибка LOOCV (на val): {loocv_val:.2f}")

# Пункт 5 из примечаний: финальное тестирование модели на отложенной выборке test и вывод ошибки
X_test_mat = np.column_stack([np.ones(len(X_test_clean)), X_test_clean.values.astype(float)])
y_test_pred = X_test_mat @ b_clean

rmse_test = np.sqrt(np.mean((y_test.values - y_test_pred) ** 2))
mae_test = np.mean(np.abs(y_test.values - y_test_pred))
r2_test = calc_r2(X_test_clean, y_test, b_clean)

print("\n--- Финальная проверка модели на тестовой выборке (test) ---")
print(f"RMSE на test: {rmse_test:.2f}")
print(f"MAE на test:  {mae_test:.2f}")
print(f"R^2 на test:  {r2_test:.6f}")

Ошибка LOOCV (на val): 12272.43

--- Финальная проверка модели на тестовой выборке (test) ---
RMSE на test: 12010.10
MAE на test:  9425.57
R^2 на test:  0.969874
